# Phase 2: baseline arm on DINOv2 features (Kaggle)

Trains three things on the extracted features and prints offline error tables:
1. **Direct policy** on frozen DINOv2 features, trained from scratch end to end.
2. **Feature-token MAE** pretraining (OctoSense-style masking over DINOv2 tokens + joints).
3. **Frozen probe** on that MAE encoder, the representation test.

Every table shows per-horizon-step mean absolute error in raw joint units next to two trivial baselines, *hold-last* and *linear extrapolation*. A policy has to beat both.

**Before running:** Add Input (right panel) → Your Work → `so101-extract` (kaggle.com/code/yashicapatodia/so101-extract) (its output has `features/grasp_1`, and `features/grasp_2` if version 2 finished). Accelerator: GPU T4 x2. Internet: On (for the git clone).

**After running:** Save Version → `/kaggle/working/outputs` holds checkpoints, logs and eval tables.

In [ ]:
GIT_REPO = "https://github.com/yashica-patodia/so101-imitation-learning.git"
LEVEL = "A"            # A = grasp (close + 0.5 s), B = grasp + lift (close + 1.5 s)
TARGET = "state"       # "state" = follower joint positions (prof: first), "action" = leader commands (later)
EPOCHS_POLICY = 20
EPOCHS_MAE = 30
EPOCHS_PROBE = 20
BATCH = 64
USE_ONLY = None        # e.g. ["grasp_1"] to restrict; None = every features/* dir found under /kaggle/input

In [ ]:
import os, glob
os.chdir("/kaggle/working")
!rm -rf /kaggle/working/repo && git clone -q {GIT_REPO} /kaggle/working/repo
!pip -q install pytest 2>&1 | tail -1
os.chdir("/kaggle/working/repo")
!python -m pytest tests -q 2>&1 | tail -1

found = sorted(glob.glob("/kaggle/input/**/meta.json", recursive=True))
found = [f for f in found if os.path.exists(os.path.join(os.path.dirname(f), "episode_000000.npz")) or glob.glob(os.path.dirname(f) + "/episode_*.npz")]
FEATURE_DIRS = [os.path.dirname(f) for f in found if USE_ONLY is None or os.path.basename(os.path.dirname(f)) in USE_ONLY]
if not FEATURE_DIRS:
    print(os.popen("ls -R /kaggle/input | head -40").read())
    raise SystemExit("no extracted features under /kaggle/input: Add Input > Your Work > so101-extract, wait for it to mount, rerun")
for d in FEATURE_DIRS:
    print(d, len(glob.glob(d + "/episode_*.npz")), "episodes")
FEATS = " ".join(FEATURE_DIRS)
OUT = "/kaggle/working/outputs"; os.makedirs(OUT, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Direct policy on DINOv2 features (from scratch)

In [ ]:
!python -m nano_vla.train.train_probe --scratch --features {FEATS} --out {OUT}/dino_policy \
    --level {LEVEL} --target {TARGET} --epochs {EPOCHS_POLICY} --batch {BATCH} 2>&1 | grep -v Warning

## 2. Feature-token MAE pretraining

In [ ]:
!python -m nano_vla.train.train_mae --features {FEATS} --out {OUT}/mae \
    --level {LEVEL} --epochs {EPOCHS_MAE} --batch {BATCH} 2>&1 | grep -v Warning

## 3. Frozen probe on the MAE encoder (representation test)

In [ ]:
!python -m nano_vla.train.train_probe --mae {OUT}/mae/best.pt --features {FEATS} --out {OUT}/mae_probe \
    --level {LEVEL} --target {TARGET} --epochs {EPOCHS_PROBE} --batch {BATCH} 2>&1 | grep -v Warning

## Summary

In [ ]:
import json, matplotlib.pyplot as plt
runs = {"dino_policy (scratch)": "dino_policy", "mae_probe (frozen)": "mae_probe"}
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, d in runs.items():
    recs = [json.loads(l) for l in open(f"{OUT}/{d}/log.jsonl")]
    ax[0].plot([r["val_mae"]["probe"] for r in recs], label=name)
    ax[0].axhline(recs[-1]["val_mae"]["hold"], ls="--", c="gray"); ax[0].axhline(recs[-1]["val_mae"]["linear"], ls=":", c="gray")
    print(f"=== {name} ===\n" + open(f"{OUT}/{d}/eval.txt").read() + "\n")
ax[0].set_title("held-out joint MAE per epoch (dashed=hold-last, dotted=linear)"); ax[0].legend(); ax[0].set_xlabel("epoch")
mrec = [json.loads(l) for l in open(f"{OUT}/mae/log.jsonl")]
ax[1].plot([r["train"]["masked"] for r in mrec], label="train masked"); ax[1].plot([r["val"]["masked"] for r in mrec], label="val masked")
ax[1].set_title("MAE reconstruction loss"); ax[1].legend(); ax[1].set_xlabel("epoch")
plt.tight_layout(); plt.show()
!du -sh {OUT}/*